In [1]:
import sys
import os
import anndata
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import anndata
import seaborn as sns
import umap
from sklearn.preprocessing import StandardScaler
%load_ext autoreload
%autoreload 2
from collections import defaultdict
import torch

In [2]:
train_adata_origin = sc.read_h5ad(f'diff_train_original.h5ad')
val_adata_origin = sc.read_h5ad(f'diff_val_original.h5ad')
test_adata_origin = sc.read_h5ad(f'diff_test_original.h5ad')
print(train_adata_origin)
print(val_adata_origin)
print(test_adata_origin)
print(train_adata_origin.obs)
print(train_adata_origin.obs["day"].unique())
print(val_adata_origin.obs["day"].unique())
print(test_adata_origin.obs["day"].unique())

AnnData object with n_obs × n_vars = 4800 × 11231
    obs: 'assigned', 'auxDir', 'cell_filter', 'cell_name', 'compatible_fragment_ratio', 'day', 'donor', 'expected_format', 'experiment', 'frag_dist_length', 'gc_bias_correct', 'is_cell_control', 'is_cell_control_bulk', 'is_cell_control_control', 'library_types', 'libType', 'log10_total_counts', 'log10_total_counts_endogenous', 'log10_total_counts_ERCC', 'log10_total_counts_feature_control', 'log10_total_counts_MT', 'log10_total_features', 'log10_total_features_endogenous', 'log10_total_features_ERCC', 'log10_total_features_feature_control', 'log10_total_features_MT', 'mapping_type', 'mates1', 'mates2', 'n_alt_reads', 'n_total_reads', 'num_assigned_fragments', 'num_bias_bins', 'num_bootstraps', 'num_compatible_fragments', 'num_consistent_mappings', 'num_inconsistent_mappings', 'num_libraries', 'num_mapped', 'num_processed', 'num_targets', 'nvars_used', 'pct_counts_endogenous', 'pct_counts_ERCC', 'pct_counts_feature_control', 'pct_counts_

In [3]:
train_adata_origin = sc.read_h5ad(f'diff_train_original.h5ad')
val_adata_origin = sc.read_h5ad(f'diff_val_original.h5ad')
test_adata_origin = sc.read_h5ad(f'diff_test_original.h5ad')

for i in [100,250,300,400]:
    # Identify the most highly variable genes
    train_adata = train_adata_origin.copy()
    val_adata = val_adata_origin.copy()
    test_adata = test_adata_origin.copy()
    sc.pp.highly_variable_genes(train_adata, n_top_genes=i)
    # others  = ['NANOG','T','GATA6']

    # for i in others:
    #     train_adata.var.highly_variable[i]=True

    var_ = train_adata.var.highly_variable
    train_adata = train_adata[:, var_]
    val_adata = val_adata[:, var_]
    test_adata = test_adata[:, var_]

    # Save datasets
    train_adata.write(f'diff_train_raw_original_gene_num_{i}.h5ad')
    val_adata.write(f'val_adata_raw_original_gene_num_{i}.h5ad')
    test_adata.write(f'test_adata_raw_original_gene_num_{i}.h5ad')

    # Print summary of each dataset
    print(f"Train dataset shape: {train_adata.shape}")
    print(f"Validation dataset shape: {val_adata.shape}")
    print(f"Test dataset shape: {test_adata.shape}")
    # data_dir = 'data/figure2_differentiation'
    train_adata = sc.read_h5ad(os.path.join(f'diff_train_raw_original_gene_num_{i}.h5ad'))
    # val_adata   = sc.read_h5ad(os.path.join(data_dir,'diff_val_raw.h5ad'))
    test_adata  = sc.read_h5ad(os.path.join(f'test_adata_raw_original_gene_num_{i}.h5ad'))
    train_adata.obs['train_data'] = 1
    test_adata.obs['train_data'] = 0

    train_adata = train_adata[train_adata.obs['day'].isin(['day0','day3'])]

    combined_adata = sc.concat([train_adata,test_adata])
    sc.tl.pca(combined_adata, svd_solver='arpack')
    sc.pp.neighbors(combined_adata, n_neighbors=30, n_pcs=50)
    sc.tl.umap(combined_adata,min_dist=0.6)

    train_adata.obs = train_adata.obs[['day']]

    train_adata.obs['day'] = train_adata.obs['day'].map({
                                                        'day0':0,
                                                        'day1':1,
                                                        'day2':2,
                                                        'day3':3                                    
                                                        })

    test_adata.obs = test_adata.obs[['day']]

    test_adata.obs['day'] = test_adata.obs['day'].map(
        {'day0':0,
        'day1':1,
        'day2':2,
        'day3':3,
    })
    train_adata_ = anndata.AnnData(train_adata.X)
    train_adata_.var_names = train_adata.var_names
    train_adata_.obs_names = train_adata.obs_names
    train_adata_.obs['Group'] = train_adata.obs['day']
    train_adata_.write(f'diff_train_gene_num_{i}.h5ad')
    test_adata_ = anndata.AnnData(test_adata.X)
    test_adata_.var_names = test_adata.var_names
    test_adata_.obs_names = test_adata.obs_names
    test_adata_.obs['Group'] = test_adata.obs['day']
    test_adata_.write(f'diff_test_gene_num_{i}.h5ad')

Train dataset shape: (4800, 100)
Validation dataset shape: (800, 100)
Test dataset shape: (2400, 100)
Train dataset shape: (4800, 250)
Validation dataset shape: (800, 250)
Test dataset shape: (2400, 250)
Train dataset shape: (4800, 300)
Validation dataset shape: (800, 300)
Test dataset shape: (2400, 300)
Train dataset shape: (4800, 400)
Validation dataset shape: (800, 400)
Test dataset shape: (2400, 400)


In [4]:
train_adata_origin = sc.read_h5ad(f'diff_train_original.h5ad')
val_adata_origin = sc.read_h5ad(f'diff_val_original.h5ad')
test_adata_origin = sc.read_h5ad(f'diff_test_original.h5ad')

for i in [200,500,1000,2000,5000,10000,11231]:
    # Identify the most highly variable genes
    train_adata = train_adata_origin.copy()
    val_adata = val_adata_origin.copy()
    test_adata = test_adata_origin.copy()
    sc.pp.highly_variable_genes(train_adata, n_top_genes=i)
    # others  = ['NANOG','T','GATA6']

    # for i in others:
    #     train_adata.var.highly_variable[i]=True

    var_ = train_adata.var.highly_variable
    train_adata = train_adata[:, var_]
    val_adata = val_adata[:, var_]
    test_adata = test_adata[:, var_]

    # Save datasets
    train_adata.write(f'diff_train_raw_original_gene_num_{i}.h5ad')
    val_adata.write(f'val_adata_raw_original_gene_num_{i}.h5ad')
    test_adata.write(f'test_adata_raw_original_gene_num_{i}.h5ad')

    # Print summary of each dataset
    print(f"Train dataset shape: {train_adata.shape}")
    print(f"Validation dataset shape: {val_adata.shape}")
    print(f"Test dataset shape: {test_adata.shape}")
    # data_dir = 'data/figure2_differentiation'
    train_adata = sc.read_h5ad(os.path.join(f'diff_train_raw_original_gene_num_{i}.h5ad'))
    # val_adata   = sc.read_h5ad(os.path.join(data_dir,'diff_val_raw.h5ad'))
    test_adata  = sc.read_h5ad(os.path.join(f'test_adata_raw_original_gene_num_{i}.h5ad'))
    train_adata.obs['train_data'] = 1
    test_adata.obs['train_data'] = 0

    train_adata = train_adata[train_adata.obs['day'].isin(['day0','day3'])]

    combined_adata = sc.concat([train_adata,test_adata])
    sc.tl.pca(combined_adata, svd_solver='arpack')
    sc.pp.neighbors(combined_adata, n_neighbors=30, n_pcs=50)
    sc.tl.umap(combined_adata,min_dist=0.6)

    train_adata.obs = train_adata.obs[['day']]

    train_adata.obs['day'] = train_adata.obs['day'].map({
                                                        'day0':0,
                                                        'day1':1,
                                                        'day2':2,
                                                        'day3':3                                    
                                                        })

    test_adata.obs = test_adata.obs[['day']]

    test_adata.obs['day'] = test_adata.obs['day'].map(
        {'day0':0,
        'day1':1,
        'day2':2,
        'day3':3,
    })
    train_adata_ = anndata.AnnData(train_adata.X)
    train_adata_.var_names = train_adata.var_names
    train_adata_.obs_names = train_adata.obs_names
    train_adata_.obs['Group'] = train_adata.obs['day']
    train_adata_.write(f'diff_train_gene_num_{i}.h5ad')
    test_adata_ = anndata.AnnData(test_adata.X)
    test_adata_.var_names = test_adata.var_names
    test_adata_.obs_names = test_adata.obs_names
    test_adata_.obs['Group'] = test_adata.obs['day']
    test_adata_.write(f'diff_test_gene_num_{i}.h5ad')

Train dataset shape: (4800, 200)
Validation dataset shape: (800, 200)
Test dataset shape: (2400, 200)
Train dataset shape: (4800, 500)
Validation dataset shape: (800, 500)
Test dataset shape: (2400, 500)
Train dataset shape: (4800, 1000)
Validation dataset shape: (800, 1000)
Test dataset shape: (2400, 1000)
Train dataset shape: (4800, 2000)
Validation dataset shape: (800, 2000)
Test dataset shape: (2400, 2000)
Train dataset shape: (4800, 5000)
Validation dataset shape: (800, 5000)
Test dataset shape: (2400, 5000)
Train dataset shape: (4800, 10000)
Validation dataset shape: (800, 10000)
Test dataset shape: (2400, 10000)
Train dataset shape: (4800, 11231)
Validation dataset shape: (800, 11231)
Test dataset shape: (2400, 11231)


In [6]:
train_adata_origin = sc.read_h5ad(f'diff_train_original.h5ad')
val_adata_origin = sc.read_h5ad(f'diff_val_original.h5ad')
test_adata_origin = sc.read_h5ad(f'diff_test_original.h5ad')
for i in [203]:
    # Identify the most highly variable genes
    train_adata = train_adata_origin.copy()
    val_adata = val_adata_origin.copy()
    test_adata = test_adata_origin.copy()
    sc.pp.highly_variable_genes(train_adata, n_top_genes=200)
    wnt_signaling = ["ADAM17",
    "AXIN1",
    "AXIN2",
    "CCND2",
    "CSNK1E",
    "CTNNB1",
    "CUL1",
    "DKK1",
    "DKK4",
    "DLL1",
    "DVL2",
    "FRAT1",
    "FZD1",
    "FZD8",
    "GNAI1",
    "HDAC11",
    "HDAC2",
    "HDAC5",
    "HEY1",
    "HEY2",
    "JAG1",
    "JAG2",
    "KAT2A",
    "LEF1",
    "MAML1",
    "MYC",
    "NCOR2",
    "NCSTN",
    "NKD1",
    "NOTCH1",
    "NOTCH4",
    "NUMB",
    "PPARD",
    "PSEN2",
    "PTCH1",
    "RBPJ",
    "SKP2",
    "TCF7",
    "TP53","WNT1","WNT5B","WNT6"]
    BMP_signaling = ["AHSG","BAMBI","BMP2","BMP4","BMP6","BMP7","BMPR1A","BMPR1B","BMPR2","CER1","CHRD","CHRDL1","CTDSP1","CTDSP2","CTDSPL","FST","GREM1","GSK3B","HJV","MAP3K7","MAPK1","NOG","NUP214","PPM1A","PPP1CA","PPP1R15A","RGMA","RGMB","SKI","SMAD1","SMAD4","SMAD5","SMAD6","SMAD7","SMAD9","SMURF1","SMURF2","SOSTDC1","TAB1","TAB2","XIAP","ZFYVE16"]
    others  = ['NANOG','T','GATA6']

    for a in others:
        train_adata.var.highly_variable[a]=True

    var_ = train_adata.var.highly_variable
    train_adata = train_adata[:, var_]
    val_adata = val_adata[:, var_]
    test_adata = test_adata[:, var_]

    # Save datasets
    train_adata.write(f'diff_train_raw_original_gene_num_{i}.h5ad')
    val_adata.write(f'val_adata_raw_original_gene_num_{i}.h5ad')
    test_adata.write(f'test_adata_raw_original_gene_num_{i}.h5ad')

    # Print summary of each dataset
    print(f"Train dataset shape: {train_adata.shape}")
    print(f"Validation dataset shape: {val_adata.shape}")
    print(f"Test dataset shape: {test_adata.shape}")
    # data_dir = 'data/figure2_differentiation'
    train_adata = sc.read_h5ad(os.path.join(f'diff_train_raw_original_gene_num_{i}.h5ad'))
    # val_adata   = sc.read_h5ad(os.path.join(data_dir,'diff_val_raw.h5ad'))
    test_adata  = sc.read_h5ad(os.path.join(f'test_adata_raw_original_gene_num_{i}.h5ad'))
    train_adata.obs['train_data'] = 1
    test_adata.obs['train_data'] = 0

    train_adata = train_adata[train_adata.obs['day'].isin(['day0','day3'])]

    combined_adata = sc.concat([train_adata,test_adata])
    sc.tl.pca(combined_adata, svd_solver='arpack')
    sc.pp.neighbors(combined_adata, n_neighbors=30, n_pcs=50)
    sc.tl.umap(combined_adata,min_dist=0.6)

    train_adata.obs = train_adata.obs[['day']]

    train_adata.obs['day'] = train_adata.obs['day'].map({
                                                        'day0':0,
                                                        'day1':1,
                                                        'day2':2,
                                                        'day3':3                                    
                                                        })

    test_adata.obs = test_adata.obs[['day']]

    test_adata.obs['day'] = test_adata.obs['day'].map(
        {'day0':0,
        'day1':1,
        'day2':2,
        'day3':3,
    })
    train_adata_ = anndata.AnnData(train_adata.X)
    train_adata_.var_names = train_adata.var_names
    train_adata_.obs_names = train_adata.obs_names
    train_adata_.obs['Group'] = train_adata.obs['day']
    train_adata_.write(f'diff_train_gene_num_{i}.h5ad')
    test_adata_ = anndata.AnnData(test_adata.X)
    test_adata_.var_names = test_adata.var_names
    test_adata_.obs_names = test_adata.obs_names
    test_adata_.obs['Group'] = test_adata.obs['day']
    test_adata_.write(f'diff_test_gene_num_{i}.h5ad')


/tmp/ipykernel_2875275/555334959.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_adata.var.highly_variable[a]=True
/tmp/ipykernel_2875275/555334959.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_adata.var.highly_variable[a]=True
/tmp/ipykernel_2875275/555334959.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_adata.var.highly_variable[a]=True


Train dataset shape: (4800, 203)
Validation dataset shape: (800, 203)
Test dataset shape: (2400, 203)
